# LoadCast: Feature Engineering & Ridge Regression Modeling
## Leakage-Free Temporal Features, Chronological Holdout Split, and Baseline Benchmarking

**Core Methodology:**
1. **Lag-1 ($y_{t-1}$):** Previous day's demand to capture immediate persistence.
2. **Lag-7 ($y_{t-7}$):** Same day last week demand to capture weekly cyclicality.
3. **Rolling 7-day Mean:** Leakage-free running average of the preceding 7 observations: $\frac{1}{7}\sum_{i=1}^{7} y_{t-i}$.
4. **Calendar Features:** Day of week ($0..6$), Month ($1..12$), Weekend indicator ($0/1$).
5. **Chronological 80/20 Train/Test Holdout Split:** Never randomly shuffle time series.
6. **Evaluation vs. Naive Persistence Baseline:** $\hat{y}_t = y_{t-1}$.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# Load processed Rajasthan data
raj = pd.read_csv('../data/processed/rajasthan_daily.csv')
raj['Date'] = pd.to_datetime(raj['Date'])

# Construct Features (Strictly Shifting to Avoid Target Leakage)
raj['lag_1'] = raj['demand'].shift(1)
raj['lag_7'] = raj['demand'].shift(7)
raj['rolling_mean_7'] = raj['demand'].shift(1).rolling(window=7, min_periods=7).mean()
raj['day_of_week'] = raj['Date'].dt.dayofweek
raj['month'] = raj['Date'].dt.month
raj['is_weekend'] = (raj['day_of_week'] >= 5).astype(int)

# Drop initial 7 rows with NaN
clean_df = raj.dropna().reset_index(drop=True)
print(f"Modeling dataset ready: {clean_df.shape[0]} rows")

### Chronological Split & Regularized Ridge Training

In [2]:
features = ['lag_1', 'lag_7', 'rolling_mean_7', 'day_of_week', 'month', 'is_weekend']
split_idx = int(len(clean_df) * 0.8)

train_df = clean_df.iloc[:split_idx]
test_df = clean_df.iloc[split_idx:]

X_train, y_train = train_df[features], train_df['demand']
X_test, y_test = test_df[features], test_df['demand']

# Fit scaler strictly on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit Ridge Regression (alpha=1.0)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)
y_pred_naive = test_df['lag_1'].values

# Compute Metrics
def eval_m(y_true, y_pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

print("Ridge Evaluation:", eval_m(y_test, y_pred_ridge))
print("Naive Baseline  :", eval_m(y_test, y_pred_naive))
print("\nFitted Coefficients:", dict(zip(features, ridge.coef_)))